# Adição de hino

Proposta de adição de hinos. A ideia é simples:

1. Criar arquivo de hino em Markdown, como se fosse na coletânea, com título como heading, e seções (Coro, Final, etc.) em negrito
2. Com arquivo de entrada, gerar SQL compatível com a estrutura do banco de dados, e adicionar a arquivo de migração

## 1. Leitura dos arquivos

Passo a passo

In [ ]:
from glob import glob

entrada = glob("arquivos_hinos\\*.md")
entrada

In [ ]:
with open(entrada[0], "r", encoding="utf-8") as f:
    conteudo = f.readlines()

conteudo

In [ ]:
titulo = None
coletanea = None
inicio_texto = 0
for i, linha in enumerate(conteudo):
    if linha.startswith("##"):
        coletanea = linha.replace("## ", "").strip()
        inicio_texto = i + 1
    elif linha.startswith("#"):
        titulo = linha.replace("# ", "").strip().upper()
        inicio_texto = i + 1

titulo, coletanea, inicio_texto

In [ ]:
texto = conteudo[inicio_texto:]
texto_full = "".join(texto).strip().replace("*", "")
print(texto_full)

In [ ]:
from pathlib import Path
import importlib.util
spec = importlib.util.spec_from_file_location("txt2json", str(Path('..') / 'etl-slides' / 'txt2json.py'))
txt2json = importlib.util.module_from_spec(spec)
spec.loader.exec_module(txt2json)
tagify_text = txt2json.tagify_text

tagify_text(texto_full)

### Função

In [ ]:
def processa_texto(arquivo: str) -> tuple[str, str, str, str]:
    with open(arquivo, "r", encoding="utf-8") as f:
        conteudo = f.readlines()

    titulo = None
    coletanea = None
    inicio_texto = 0
    for i, linha in enumerate(conteudo):
        if linha.startswith("##"):
            coletanea = linha.replace("## ", "").strip()
            inicio_texto = i + 1
        elif linha.startswith("#"):
            titulo = linha.replace("# ", "").strip().upper()
            inicio_texto = i + 1

    texto = conteudo[inicio_texto:]
    texto = "".join(texto).strip().replace("*", "")
    texto_processado = tagify_text(texto)

    return titulo, coletanea, texto, texto_processado

In [ ]:
processa_texto(entrada[1])

## 2. Processamento do texto em SQL

In [ ]:
INSERT_COLUMNS = "numero, nome, nome_pt, texto, texto_processado, coletanea_id, idioma, date_insert, date_update"
arquivo_migracao = "010-add-hinos-avulsos.sql"

with open("..\\..\\database\\migrations\\" + arquivo_migracao, "w", encoding="utf-8") as f:
    for arquivo in entrada:
        titulo, coletanea, texto, texto_processado = processa_texto(arquivo)

        linha = (
            "INSERT INTO hino ("
            + INSERT_COLUMNS
            + ") VALUES (NULL,'"
            + titulo
            + "','"
            + titulo
            + "','"
            + texto.replace("\n", "\\n").replace("'", "''")
            + "','"
            + texto_processado.replace("\n", "\\n").replace("'", "''")
            + "',(select id from coletanea where nome = '"
            + coletanea
            + "'),'PT-BR',CURRENT_TIMESTAMP,CURRENT_TIMESTAMP);\n"
        )

        f.write(linha)